# A linear probe on pooled token features

A frozen text encoder gives one feature vector per token. To classify whole reviews as
positive or negative with a linear probe, each review needs one vector, and the usual
choice is the average of its token vectors. The reviews come in batches padded to the
longest review, and the encoder outputs a vector for every column, pads included. You
will pool the features and train the probe, a logistic regression head, in NumPy.

**How to work through it.** Each part asks for one function. Replace the `TODO` and the
`raise NotImplementedError` with your code, then run the check cell under it. The check
prints `Part N passed` when your function is right, and stops with a message that says
what is wrong when it is not.

## Setup

Run this cell first. The training reviews have 4 to 12 words and the test reviews 4 to 40.

In [ ]:
import numpy as np

# Word ids 1 to 8 are positive words, 9 to 16 negative, 17 to 39 neutral. Id 0 is the pad.
V, D_EMB, D, PAD = 40, 16, 24, 0
rng = np.random.default_rng(11)
polarity = np.zeros(V)
polarity[1:9], polarity[9:17] = 1.0, -1.0
E = rng.normal(0, 1, (V, D_EMB))
E[:, 0] += 2.0 * polarity                      # pretraining left a sentiment direction in the embeddings
W_enc = rng.normal(0, 1 / np.sqrt(D_EMB), (D_EMB, D))


def encode(ids):
    """The frozen encoder: one D-dimensional feature vector per token, pads included."""
    return np.tanh(E[ids] @ W_enc)


def make_reviews(n, min_len, max_len, seed):
    """n reviews as lists of word ids, labeled 1 when they hold more positive than negative words."""
    r = np.random.default_rng(seed)
    reviews, labels = [], []
    while len(reviews) < n:
        words = r.integers(1, V, int(r.integers(min_len, max_len + 1)))
        score = polarity[words].sum()
        if score != 0:
            reviews.append([int(w) for w in words])
            labels.append(int(score > 0))
    return reviews, np.array(labels)


def pad_right(reviews):
    """Ids padded on the right to the longest review, and the mask (1 for a real token)."""
    T = max(len(r) for r in reviews)
    ids = np.full((len(reviews), T), PAD)
    mask = np.zeros((len(reviews), T), dtype=int)
    for i, r in enumerate(reviews):
        ids[i, :len(r)] = r
        mask[i, :len(r)] = 1
    return ids, mask


train_reviews, y_train = make_reviews(400, 4, 12, seed=1)      # short reviews
test_reviews, y_test = make_reviews(400, 4, 40, seed=2)        # some much longer
ids_train, mask_train = pad_right(train_reviews)
ids_test, mask_test = pad_right(test_reviews)
H_train, H_test = encode(ids_train), encode(ids_test)          # the encoder runs once
print("token features:", H_train.shape, H_test.shape)

## Part 1 (coding task): One vector per review

Write `mean_pool(H, mask)`. `H` has shape `(n, T, D)`, one feature vector per column, and
`mask` has shape `(n, T)`, 1 for a real word and 0 for a pad. Return an `(n, D)` array
whose row $i$ is the average of the feature vectors of review $i$'s words.

In [ ]:
def mean_pool(H, mask):
    """One vector per review, shape (n, D): the average of the review's token features."""
    # TODO
    raise NotImplementedError

In [ ]:
review = [3, 20, 11, 25, 5]
ids_a, mask_a = pad_right([review, [17] * 6])                  # padded to 6 columns
ids_b, mask_b = pad_right([review, [17] * 30])                 # the same review padded to 30
pooled_a, pooled_b = mean_pool(encode(ids_a), mask_a), mean_pool(encode(ids_b), mask_b)
assert np.shape(pooled_a) == (2, D), f"expected one row of {D} numbers per review, got shape {np.shape(pooled_a)}"
want = encode(np.array([review]))[0].mean(axis=0)
assert np.allclose(pooled_b[0], pooled_a[0]), (
    "the same review pools to a different vector when its batch is padded to 30 columns instead of 6")
assert np.allclose(pooled_a[0], want), "the pooled vector should be the mean of the review's own 5 token vectors"
F_check = mean_pool(H_train, mask_train)
assert np.allclose(F_check[0], H_train[0, :len(train_reviews[0])].mean(axis=0)), "wrong pooled vector for the first training review"
print("Part 1 passed")

## Part 2 (coding task): Train the probe

Write `train_probe(F, y, steps, lr)`: logistic regression on the fixed features `F`, with
weights `w` of shape `(D,)` and a bias `b`. With $p_i = \sigma(F_i \cdot w + b)$, row $i$
costs $-\log p_i$ when $y_i = 1$ and $-\log(1 - p_i)$ when $y_i = 0$, and the probe
minimizes the average cost over the $n$ rows. Start from `w = 0` and `b = 0` and take
`steps` gradient steps of size `lr`, each using all the rows. The check then pools the
reviews, trains the probe and measures it on the test reviews.
The seeded run reaches a test accuracy of about 0.90, and the check below asks for 0.80,
because the exact value moves with the encoder's seed.


In [ ]:
def train_probe(F, y, steps, lr):
    """Logistic regression on fixed features F (n, D), trained by gradient descent on all rows.

    Start from w = 0 and b = 0, and return (w, b) after `steps` steps.
    """
    # TODO
    raise NotImplementedError

In [ ]:
F4 = np.array([[1.0, 2.0], [0.0, -1.0], [2.0, 0.0], [-1.0, 1.0]])
w1, b1 = train_probe(F4, np.array([1, 0, 1, 0]), steps=1, lr=0.5)
assert np.allclose(w1, [0.25, 0.125]) and np.isclose(b1, 0.0), (
    f"one step from zero should give w = (0.25, 0.125) and b = 0, got w = {np.round(w1, 4)}, b = {np.round(b1, 4)}: "
    "a step is lr times the gradient of the mean loss over the four rows, downhill")
w2, b2 = train_probe(F4, np.array([1, 1, 1, 0]), steps=1, lr=0.5)
assert np.allclose(w2, [0.25, 0.0]) and np.isclose(b2, 0.125), (
    f"with labels (1, 1, 1, 0) one step should give w = (0.25, 0) and b = 0.125, got w = {np.round(w2, 4)}, b = {np.round(b2, 4)}")
w3, b3 = train_probe(F4, np.array([1, 1, 1, 0]), steps=200, lr=0.5)
assert np.allclose(w3, [4.3792, -1.3274], atol=1e-3) and np.isclose(b3, 2.1291, atol=1e-3), (
    f"after 200 steps expected w = (4.3792, -1.3274) and b = 2.1291, got w = {np.round(w3, 4)}, b = {np.round(b3, 4)}")

F_train, F_test = mean_pool(H_train, mask_train), mean_pool(H_test, mask_test)
w, b = train_probe(F_train, y_train, steps=500, lr=0.5)
acc_train = np.mean((F_train @ w + b > 0) == y_train)
acc_test = np.mean((F_test @ w + b > 0) == y_test)
wn, bn = train_probe(H_train.mean(axis=1), y_train, steps=500, lr=0.5)
acc_all_columns = np.mean((H_test.mean(axis=1) @ wn + bn > 0) == y_test)
print(f"probe on {D} frozen features: train accuracy {acc_train:.3f}, test accuracy {acc_test:.3f}")
print(f"the same probe on averages over every column, pads included: test accuracy {acc_all_columns:.3f}")
assert acc_test > 0.80, f"test accuracy {acc_test:.3f}: expected above 0.80 from a correct pool and probe"
print("Part 2 passed")

## Solution

### Part 1: One vector per review

In [ ]:
def mean_pool(H, mask):
    """One vector per review, shape (n, D): the average of the review's token features."""
    m = np.asarray(mask, dtype=float)[:, :, None]           # (n, T, 1)
    return (H * m).sum(axis=1) / m.sum(axis=1)

**Why it works.** The encoder gives the pad a feature vector like any other token, `tanh(E[0] @ W_enc)`,
which is not zero. An average over all $T$ columns mixes in that vector with weight
$(T - n)/T$ for a review of $n$ words, so the same review moves when its batch is padded
further, and it moves more in a batch with one long review. Multiplying by the mask
removes the pads from the sum, and dividing by the mask's sum divides by $n$, not $T$.

### Part 2: Train the probe

In [ ]:
def train_probe(F, y, steps, lr):
    """Logistic regression on fixed features F (n, D), trained by gradient descent on all rows.

    Start from w = 0 and b = 0, and return (w, b) after `steps` steps.
    """
    F, y = np.asarray(F, dtype=float), np.asarray(y, dtype=float)
    w, b = np.zeros(F.shape[1]), 0.0
    for _ in range(steps):
        p = 1 / (1 + np.exp(-(F @ w + b)))
        err = p - y                               # n times the derivative of the average cost in each logit
        w = w - lr * F.T @ err / len(y)
        b = b - lr * err.mean()
    return w, b

**Why it works.** For the average log loss the derivative in the logit of row $i$ is
$(p_i - y_i)/n$, so the weights move by $-\text{lr}\,F^\top(p - y)/n$ and the bias by
$-\text{lr}$ times the mean of $p - y$. In the first check every $p_i$ starts at 0.5, so the
first step is $0.5 \cdot F^\top(y - 0.5)/4 = (0.25, 0.125)$. The encoder ran once, before any
training, and only the $D + 1 = 25$ numbers of the head were ever updated: that is a
linear probe, and its test accuracy measures how much of the sentiment the frozen
features already hold. The last comparison is the cost of Part 1's mistake: the training
reviews are all short, the test batch is padded to 40 columns, and the pad's vector
shifts every short test review's average to a place the probe never saw in training.